# Lección 12.5 — Por qué los Transformers escalan tan bien: leyes de escalamiento

> **Módulo 12 · Atención y Transformers** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 12.3 (bloque Transformer), Lección 12.4 (decoders), Lección 4.4 (sobreajuste vs. subajuste)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué los Transformers aprovechan mejor que las RNN el hardware moderno.
- Leer una gráfica **log-log** y reconocer una **ley de potencia** (una recta en esa gráfica).
- Hacer **tu propio experimento de escalamiento**: entrenar Transformers de varios tamaños y con distintas cantidades de datos, y ajustar la recta.
- Usar la regla de **Chinchilla** (≈20 tokens por parámetro) para planear un presupuesto de entrenamiento.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> ⏱️ Este notebook **entrena más de 15 modelos pequeños**. En CPU, todo tarda unos 2–3 minutos.

## ▶️ Preparación
Esta celda descarga el *Quijote* completo (≈2 MB, dominio público, Proyecto Gutenberg), lo pasa a minúsculas y lo convierte en números (un número por carácter).

In [ ]:
import os, re, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

plt.rcParams["figure.figsize"] = (6.5, 4.5)
device = "cuda" if torch.cuda.is_available() else "cpu"

FUENTES = ["https://gutenberg.pglaf.org/2/0/0/2000/2000-0.txt",
           "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"]

def cargar_quijote(archivo="quijote.txt"):
    if not os.path.exists(archivo):
        for url in FUENTES:
            try:
                urllib.request.urlretrieve(url, archivo)
                break
            except Exception as error:
                print("No se pudo descargar de", url, "→", error)
    texto = open(archivo, encoding="utf-8").read()
    inicio = texto.find("***", texto.find("*** START") + 3) + 3        # quitar el encabezado de Gutenberg
    texto = texto[inicio:texto.find("*** END")]
    return re.sub(r"\s+", " ", texto.lower())                          # minúsculas y espacios simples

texto = cargar_quijote()
letras = sorted(set(texto))
a_numero = {c: i for i, c in enumerate(letras)}
datos = torch.tensor([a_numero[c] for c in texto])
corte = int(0.9 * len(datos))
datos_train, datos_val = datos[:corte], datos[corte:]       # 90 % para entrenar, el último 10 % para validar
V = len(letras)
print(f"✅ {len(texto):,} caracteres · {V} caracteres distintos · dispositivo: {device}")
print("Muestra:", texto[1000:1150])

---
## 1. 📖 La intuición: la cadena de cubetas vs. la página entera

Imagina que hay que apagar un incendio con agua de un río:

- 🪣 **RNN (Lección 11.3) = cadena de cubetas.** Cada persona recibe la cubeta del anterior, y la pasa. No importa cuántas personas contrates: la cubeta **tiene que pasar por todas, una tras otra**. Para leer el token 1,000, primero hay que procesar los 999 anteriores.
- 🚿 **Transformer = una manguera por persona.** Todas trabajan **a la vez**. Cada token mira a todos los demás en **un solo paso** de atención (una gran multiplicación de matrices).

Las GPU (y los chips especializados) son máquinas de hacer **muchísimas multiplicaciones en paralelo**. Los Transformers les dan exactamente eso. Por eso, al poner más hardware, un Transformer se entrena **más rápido y con más datos**, mientras que una RNN queda limitada por su cadena.

| | RNN / LSTM | Transformer |
|---|---|---|
| Pasos **secuenciales** para leer *n* tokens | *n* | 1 por capa |
| Distancia entre el token 1 y el token *n* | *n* pasos | 1 paso |
| Entrenar las *n* predicciones de una frase | Una tras otra | Todas a la vez (máscara causal, Lección 12.4) |
| Costo por capa | Crece como *n* | Crece como *n²* (pero en paralelo) |

### ✋ Ejercicio 1 — Cuenta los pasos
Un texto tiene **2,048 tokens**. Un modelo tiene **12 capas**.
1. ¿Cuántos pasos **secuenciales** (uno tras otro, sin poder paralelizar) necesita una RNN de 12 capas para leerlo, aproximadamente?
2. ¿Y un Transformer de 12 capas?
3. Si tuvieras 1,000 GPU, ¿a cuál de los dos le ayudarían más?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Del orden de **2,048** pasos secuenciales (cada token espera al anterior; las capas pueden ir en cadena detrás, pero el recorrido sobre los tokens no se puede saltar).
2. **12** pasos: una capa tras otra, pero cada capa procesa los 2,048 tokens a la vez.
3. Al **Transformer**: tiene muchísimo trabajo independiente que repartir. A la RNN, más GPUs casi no le sirven para recorrer *una* frase más rápido.

Esta ventaja práctica, sumada a que **mejora de forma predecible al crecer** (lo que verás ahora), explica por qué los modelos pasaron de millones a cientos de miles de millones de parámetros en pocos años.
</details>

---
## 2. 📖 El concepto: las leyes de potencia

En 2020, un equipo de OpenAI (Kaplan y colegas) entrenó cientos de Transformers de tamaños muy distintos y descubrió algo sorprendente: la pérdida baja de forma **muy regular** cuando aumentas:
- **N**: el número de parámetros del modelo,
- **D**: la cantidad de datos (tokens) de entrenamiento,
- **C**: el cómputo total (operaciones) usado para entrenar.

La forma de esa mejora es una **ley de potencia**:

```
pérdida ≈ a · N^(−α)          (y lo mismo con D o con C)
```

En palabras: **cada vez que multiplicas el tamaño por un mismo factor, la pérdida baja en el mismo porcentaje**. No es "un poco más de modelo, un poco menos de pérdida" (lineal), sino "×10 de modelo, −X % de pérdida", una y otra vez.

El truco para verlas: dibujarlas en una gráfica **log-log** (ambos ejes en escala logarítmica, donde cada marca vale 10 veces la anterior).

### 🔮 Ejercicio 2 — ¿Curva o recta?
Vamos a dibujar `pérdida = 5 · N^(−0.3)` para N entre 1,000 y 1,000,000,000. ¿Cómo se verá en ejes normales? ¿Y en ejes log-log?

✍️ *Mi predicción:*

In [ ]:
N = np.logspace(3, 9, 50)          # de mil a mil millones
perdida = 5 * N ** -0.3

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(N, perdida); axes[0].set_title("Ejes normales")
axes[1].loglog(N, perdida); axes[1].set_title("Ejes log-log")
for ax in axes:
    ax.set_xlabel("N (parámetros)"); ax.set_ylabel("pérdida"); ax.grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Ejes normales:** una curva que cae casi en vertical al principio y luego parece **plana**. Engañosa: parece que más allá de cierto tamaño ya no se gana nada.
- **Log-log:** una **recta** perfecta. Su **pendiente** es el exponente (−0.3).

Por eso todas las gráficas de escalamiento se dibujan en log-log: una recta significa "la mejora sigue llegando, al mismo ritmo, por cada multiplicación del tamaño".
</details>

### 🔮 Ejercicio 3 — ¿Cuánto hay que crecer?
Kaplan y colegas midieron, para Transformers de lenguaje, un exponente **α ≈ 0.076** para el número de parámetros. Es un número pequeño. ¿Cuántas veces hay que multiplicar el tamaño del modelo para que la pérdida baje un **10 %**?

✍️ *Mi apuesta:* ×___

In [ ]:
alfa = 0.076                     # ✏️ prueba con el exponente de los datos: 0.095
for mejora in [0.10, 0.20, 0.50]:
    factor = (1 - mejora) ** (-1 / alfa)
    print(f"Bajar la pérdida un {mejora:.0%} requiere un modelo ×{factor:,.0f} más grande")

<details><summary><b>👀 Ver respuesta</b></summary>

- −10 % → **×4** parámetros.
- −20 % → **×19**.
- −50 % → ¡**×9,000** aproximadamente!

Cada mejora cuesta **multiplicar**, no sumar. Esto explica la carrera por modelos gigantes: la mejora es **predecible** (sabes lo que vas a obtener antes de gastar), pero cada vez **más cara**.
</details>

---
## 3. 📖 Tu propio experimento: ¿qué pasa al agrandar el modelo?

Vamos a comprobar la ley con nuestras manos. Entrenaremos **mini-GPTs de caracteres** (un decoder como el de la Lección 12.4) sobre el *Quijote*, con distintos tamaños de `d_model`, y **el mismo número de pasos** para todos. Mediremos la **pérdida de validación** (entropía cruzada, Lección 3.2, sobre el último 10 % del libro, que el modelo nunca ve al entrenar).

In [ ]:
CONTEXTO, LOTE = 32, 32
mascara = torch.triu(torch.ones(CONTEXTO, CONTEXTO, dtype=torch.bool, device=device), diagonal=1)

class Bloque(nn.Module):                       # el bloque de la Lección 12.3, con máscara causal
    def __init__(self, d, h):
        super().__init__()
        self.norma1, self.norma2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.atencion = nn.MultiheadAttention(d, h, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        h = self.norma1(x)
        x = x + self.atencion(h, h, h, attn_mask=mascara, need_weights=False)[0]
        return x + self.mlp(self.norma2(x))

class MiniGPT(nn.Module):
    def __init__(self, d_model, n_cabezas=2, n_capas=2):
        super().__init__()
        self.embedding = nn.Embedding(V, d_model)
        self.posicion = nn.Embedding(CONTEXTO, d_model)
        self.bloques = nn.Sequential(*[Bloque(d_model, n_cabezas) for _ in range(n_capas)])
        self.norma = nn.LayerNorm(d_model)
        self.salida = nn.Linear(d_model, V)
    def forward(self, x):
        z = self.embedding(x) + self.posicion(torch.arange(x.shape[1], device=x.device))
        return self.salida(self.norma(self.bloques(z)))

def lote(fuente, n=LOTE, generador=None):
    i = torch.randint(0, len(fuente) - CONTEXTO - 1, (n,), generator=generador)
    x = torch.stack([fuente[j:j + CONTEXTO] for j in i])
    y = torch.stack([fuente[j + 1:j + CONTEXTO + 1] for j in i])
    return x.to(device), y.to(device)

@torch.no_grad()
def perdida_val(modelo):
    modelo.eval()
    x, y = lote(datos_val, 512, torch.Generator().manual_seed(1))     # siempre los mismos 512 fragmentos
    p = nn.functional.cross_entropy(modelo(x).reshape(-1, V), y.reshape(-1)).item()
    modelo.train()
    return p

def entrenar(d_model, fuente=datos_train, pasos=500):
    """Entrena un MiniGPT y devuelve (n_parámetros, historial de (pasos, pérdida de validación))."""
    torch.manual_seed(0)
    modelo = MiniGPT(d_model).to(device)
    opt = torch.optim.AdamW(modelo.parameters(), lr=3e-3)
    historial = []
    for paso in range(1, pasos + 1):
        x, y = lote(fuente)
        perdida = nn.functional.cross_entropy(modelo(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); perdida.backward(); opt.step()
        if paso % 50 == 0:
            historial.append((paso, perdida_val(modelo)))
    return sum(p.numel() for p in modelo.parameters()), historial

print("✅ Listo para experimentar")

### 🔮 Ejercicio 4 — Predice la forma
Entrenaremos 4 tamaños: `d_model` = 8, 16, 32 y 64 (≈3 mil a ≈110 mil parámetros), 500 pasos cada uno.
1. ¿La pérdida bajará siempre al agrandar el modelo?
2. En la gráfica log-log, ¿esperas una recta, una curva que se aplana, o puntos al azar?

✍️ *Mis predicciones:*

In [ ]:
TAMANOS = [8, 16, 32, 64]
resultados = {}
inicio = time.time()
for d in TAMANOS:
    n_params, hist = entrenar(d)
    resultados[d] = (n_params, hist)
    print(f"d_model = {d:>3} → {n_params:>7,} parámetros · pérdida de validación final = {hist[-1][1]:.3f}")
print(f"({time.time() - inicio:.0f} s)")

N_exp = np.array([resultados[d][0] for d in TAMANOS])
L_exp = np.array([resultados[d][1][-1][1] for d in TAMANOS])
pendiente, intercepto = np.polyfit(np.log(N_exp), np.log(L_exp), 1)      # ajustar una recta en log-log
print(f"Pendiente de la recta en log-log: {pendiente:.3f}")

plt.loglog(N_exp, L_exp, "o", ms=8, label="experimento")
N_linea = np.logspace(np.log10(N_exp.min()), np.log10(N_exp.max()), 50)
plt.loglog(N_linea, np.exp(intercepto) * N_linea ** pendiente, "--", label=f"ley de potencia (pendiente {pendiente:.3f})")
plt.xlabel("parámetros (N)"); plt.ylabel("pérdida de validación"); plt.grid(alpha=.3, which="both")
plt.legend(); plt.title("Más parámetros → menos pérdida"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sí**: de ≈2.35 (d = 8) a ≈1.87 (d = 64). Cada vez que el modelo crece ≈×3.5, la pérdida baja.
2. Los puntos quedan **muy cerca de una recta** en log-log, con una pendiente de ≈**−0.064**: tu propio α ≈ 0.06, del mismo orden que el 0.076 de Kaplan, ¡con modelos un millón de veces más pequeños y en unos segundos de CPU!

(Con tan pocos puntos y modelos tan pequeños, la recta es solo aproximada. Los equipos que estudian esto entrenan cientos de modelos.)
</details>

---
## 4. 📖 ¿Qué pasa si…? Predecir un modelo antes de entrenarlo

Aquí está el valor práctico de las leyes de escalamiento: antes de gastar millones en un modelo enorme, los laboratorios **entrenan modelos pequeños, ajustan la recta y extrapolan**. Por ejemplo, el informe técnico de GPT-4 cuenta que predijeron su pérdida final con modelos que usaban hasta 10,000 veces menos cómputo.

Hagamos lo mismo: con la recta del experimento, **predice** la pérdida de un modelo con `d_model = 128` (≈4 veces más parámetros que el más grande que entrenaste) y luego entrénalo para comprobarlo.

### 🔮 Ejercicio 5 — Extrapola y comprueba
Antes de ejecutar: ¿crees que la predicción acertará? ¿Será la pérdida real **mejor** (más baja) o **peor** (más alta) que lo predicho?

✍️ *Mi predicción:*

In [ ]:
n_128, hist_128 = entrenar(128)
resultados[128] = (n_128, hist_128)
prediccion = np.exp(intercepto) * n_128 ** pendiente

print(f"d_model = 128 → {n_128:,} parámetros")
print(f"Pérdida PREDICHA por la recta: {prediccion:.3f}")
print(f"Pérdida REAL                 : {hist_128[-1][1]:.3f}")

plt.loglog(N_exp, L_exp, "o", ms=8, label="experimento (para ajustar)")
plt.loglog([n_128], [hist_128[-1][1]], "r*", ms=15, label="d_model = 128 (real)")
N_linea = np.logspace(np.log10(N_exp.min()), np.log10(n_128 * 1.5), 50)
plt.loglog(N_linea, np.exp(intercepto) * N_linea ** pendiente, "--", label="extrapolación")
plt.xlabel("parámetros (N)"); plt.ylabel("pérdida de validación"); plt.grid(alpha=.3, which="both")
plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La recta predice ≈**1.73** y el modelo real obtiene ≈**1.78**: un error de solo ≈3 %, pero **peor** de lo predicho. Dos razones:
1. **Pérdida irreducible:** ningún modelo puede bajar de cierto valor (el texto tiene una parte impredecible). La ley completa es `pérdida ≈ irreducible + a·N^(−α)`, así que la recta pura es demasiado optimista a medida que te acercas al piso.
2. **El modelo grande quedó corto de entrenamiento:** todos entrenaron **500 pasos**. Un modelo más grande necesita **más datos y más pasos** para aprovechar su tamaño. Eso es justo lo siguiente que vamos a estudiar.
</details>

---
## 5. 📖 El otro eje: más datos

Ahora fijamos el modelo y cambiamos **cuántos datos** puede usar. Entrenaremos con los primeros 3 mil, 10 mil, 30 mil, 100 mil y 1 millón de caracteres del libro, siempre 500 pasos, y con **dos tamaños** de modelo: uno chico (`d_model = 16`) y uno mediano (`d_model = 64`).

Como con pocos datos el modelo puede **sobreajustar** (Lección 4.4), guardamos la **mejor** pérdida de validación del entrenamiento (como si usáramos *early stopping*, Lección 7.2).

### 🔮 Ejercicio 6 — ¿Más datos siempre ayuda?
1. ¿Qué le pasará a la pérdida con 3 mil caracteres?
2. ¿Esperas que las curvas sigan bajando hasta el millón de caracteres, o que se aplanen en algún momento? ¿Cuál se aplana antes, la del modelo chico o la del mediano?

✍️ *Mis predicciones:*

In [ ]:
TAMANOS_DATOS = [3_000, 10_000, 30_000, 100_000, 1_000_000]    # ✏️
inicio = time.time()
curvas = {}
for d in [16, 64]:
    curvas[d] = []
    for n_car in TAMANOS_DATOS:
        _, hist = entrenar(d, fuente=datos_train[:n_car])
        mejor = min(p for _, p in hist)
        curvas[d].append(mejor)
        print(f"d_model = {d:>2} · {n_car:>9,} caracteres → mejor pérdida de validación {mejor:.3f}")
print(f"({time.time() - inicio:.0f} s)")

for d, marca in [(16, "o-"), (64, "s-")]:
    plt.loglog(TAMANOS_DATOS, curvas[d], marca, label=f"d_model = {d}")
plt.xlabel("caracteres de entrenamiento (D)"); plt.ylabel("mejor pérdida de validación")
plt.grid(alpha=.3, which="both"); plt.legend(); plt.title("Más datos → menos pérdida… hasta cierto punto"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con 3 mil caracteres, la pérdida es mala (≈2.5) para **los dos** modelos: el modelo memoriza ese pedacito y no generaliza.
2. Las dos curvas bajan rápido al principio y luego **se aplanan**:
   - El **chico** (d = 16) se estanca pronto (≈2.25 desde los 100 mil caracteres): ya no tiene **capacidad** para aprender más, aunque le des más datos.
   - El **mediano** (d = 64) sigue mejorando más tiempo y llega más abajo (≈1.88), pero también se aplana: con 500 pasos de 32 fragmentos de 32 caracteres solo alcanza a **ver** ≈512 mil caracteres en total, así que el millón no le sirve de mucho. Ahí el cuello de botella es el **cómputo**.

La gran lección: **escalar una sola cosa se acaba**. Datos sin modelo (o modelo sin datos, o ambos sin cómputo) llegan a un techo. Hay que escalar **todo junto**.
</details>

---
## 6. 📖 El cómputo y la regla de Chinchilla

Para un Transformer, el **cómputo** de entrenamiento se puede estimar con una regla sencilla:

```
C ≈ 6 · N · D       (operaciones = 6 × parámetros × tokens vistos)
```

Si tienes un presupuesto fijo de cómputo, ¿conviene un modelo **grande** entrenado con **pocos** datos, o uno **chico** con **muchos**? Usemos los historiales del experimento de la Sección 3: para cada modelo y cada momento del entrenamiento, calculamos su cómputo acumulado.

### 🔮 Ejercicio 7 — Con el mismo presupuesto…
Con un presupuesto **pequeño** de cómputo, ¿quién crees que tendrá menos pérdida: el modelo más grande (d = 128) o uno mediano?

✍️ *Mi predicción:*

In [ ]:
for d in [8, 16, 32, 64, 128]:
    n_params, hist = resultados[d]
    pasos = np.array([p for p, _ in hist])
    computo = 6 * n_params * pasos * LOTE * CONTEXTO
    plt.loglog(computo, [l for _, l in hist], "-", label=f"d = {d} ({n_params:,} parámetros)")
plt.xlabel("cómputo usado (operaciones, C ≈ 6·N·D)"); plt.ylabel("pérdida de validación")
plt.grid(alpha=.3, which="both"); plt.legend(fontsize=8)
plt.title("Cada curva es un tamaño de modelo durante su entrenamiento"); plt.show()

presupuesto = 1e11
for d in [8, 16, 32, 64, 128]:
    n_params, hist = resultados[d]
    alcanzables = [l for p, l in hist if 6 * n_params * p * LOTE * CONTEXTO <= presupuesto]
    print(f"Con 10^11 operaciones, d = {d:>3} llega a: " + (f"{alcanzables[-1]:.3f}" if alcanzables else "no alcanza ni 50 pasos"))

<details><summary><b>👀 Ver respuesta</b></summary>

Con un presupuesto de 10¹¹ operaciones, el que mejor queda es el **mediano** (d = 32, ≈2.06), por delante del de d = 64 (≈2.23), mientras que el grande (d = 128) **ni siquiera alcanza** a hacer 50 pasos. Las curvas **se cruzan**: con poco cómputo conviene un modelo más chico entrenado más tiempo; con más cómputo, conviene uno más grande. La línea que une los mejores puntos de cada presupuesto es la **frontera eficiente**.

En 2022, DeepMind (Hoffmann y colegas, el modelo **Chinchilla**) estudió esta frontera a gran escala y llegó a una regla práctica:

> 🐭 **Regla de Chinchilla:** para usar bien el cómputo, entrena con unos **20 tokens por cada parámetro** (D ≈ 20·N).

Con eso descubrieron que muchos modelos de la época eran demasiado grandes para los datos que vieron: GPT-3 (175 mil millones de parámetros) se entrenó con 300 mil millones de tokens, menos de 2 por parámetro. Chinchilla, con 70 mil millones de parámetros y 1.4 billones de tokens, superó a modelos 4 veces más grandes.
</details>

### ✋ Ejercicio 8 — Aplica Chinchilla
Calcula (en papel o en la celda) cuántos tokens recomienda Chinchilla para un modelo de **7 mil millones** de parámetros, y cuánto cómputo (C ≈ 6·N·D) costaría.

In [ ]:
N_modelo = 7e9                     # ✏️ prueba 70e9 o el tamaño de GPT-2 (124e6)
D_optimo = 20 * N_modelo
C = 6 * N_modelo * D_optimo
print(f"Tokens recomendados: {D_optimo:.1e}  ({D_optimo / 1e9:,.0f} mil millones)")
print(f"Cómputo: {C:.1e} operaciones")

<details><summary><b>👀 Ver respuesta</b></summary>

- Tokens: **1.4 × 10¹¹** (140 mil millones).
- Cómputo: ≈ **5.9 × 10²¹** operaciones.

Dato curioso: los modelos abiertos modernos suelen entrenarse con **muchos más** tokens que los de Chinchilla. Por ejemplo, Llama 3 de 8 mil millones de parámetros se entrenó con más de 15 billones de tokens (≈1,900 por parámetro). ¿Por qué "desperdiciar" cómputo? Porque Chinchilla optimiza **el costo de entrenar**, pero un modelo se usa millones de veces: un modelo **más chico** y sobre-entrenado es más barato de **usar** (Lección 15.1). Las leyes de escalamiento sirven para decidir, no son un dogma.
</details>

---
## 7. 📖 Las perillas del escalamiento

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Regla de partida |
|---|---|---|---|---|
| **N** (parámetros) | Capacidad del modelo | Menos pérdida, si hay datos y cómputo suficientes; más caro de usar | Más barato; techo de capacidad más bajo | Decídelo junto con D |
| **D** (tokens de entrenamiento) | Cuánto "mundo" ve el modelo | Menos pérdida, si el modelo tiene capacidad; menos sobreajuste | Riesgo de sobreajuste/memorización | D ≈ 20·N (Chinchilla) o más si el modelo se usará mucho |
| **C** (cómputo) | Presupuesto total (≈ 6·N·D) | Permite subir N y D a la vez | Obliga a elegir: modelo chico bien entrenado | Repártelo según la frontera eficiente |
| Tasa de aprendizaje | Tamaño del paso (Lección 9.6) | Inestable en modelos grandes | Lento | Los modelos más grandes suelen necesitar una tasa **más baja** |
| Tamaño de lote | Tokens por paso | Más paralelo y estable | Más ruido, más pasos | Crece con el tamaño del modelo |

### 🔮 Ejercicio 9 — ¿Todo mejora igual?
La pérdida baja suave y predecible. ¿Significa eso que **todas** las habilidades del modelo (sumar números, traducir, resolver acertijos…) mejoran también de forma suave?

✍️ *Mi opinión:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No necesariamente.** Algunas habilidades medidas con exactitud de "todo o nada" parecen aparecer de golpe a partir de cierto tamaño: se les llama **habilidades emergentes**. Hay un debate abierto: varios estudios muestran que, si se miden con una métrica más suave (por ejemplo, cuántos dígitos acierta en vez de si la suma completa está bien), la mejora vuelve a verse gradual. Moraleja: **la métrica que eliges cambia la historia que cuentas** (Lección 4.8). Lo que sí es predecible es la **pérdida**.
</details>

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| Comparar modelos de distinto tamaño con distinto entrenamiento | La diferencia puede deberse al entrenamiento, no al tamaño | Fijar pasos/datos, o comparar a igual cómputo |
| Ajustar una recta con 2–3 puntos y extrapolar ×1,000 | Ruido + pérdida irreducible = predicciones muy erróneas | Muchos puntos, varios órdenes de magnitud, modelo con término irreducible |
| "Más grande siempre es mejor" | Sin datos ni cómputo suficientes, el modelo grande queda corto (Ejercicio 7) | Escalar N, D y C juntos |
| Dibujar en ejes lineales | La ley parece "aplanarse" y engaña (Ejercicio 2) | Usar log-log |
| Olvidar el costo de **usar** el modelo | Un modelo enorme puede ser impagable en producción | Considerar modelos más chicos sobre-entrenados (Ejercicio 8) |
| Medir en los datos de entrenamiento | Un modelo que memoriza parece escalar de maravilla | Medir siempre en validación (Lección 2.5) |

## 9. 📋 Ficha: leyes de escalamiento

| | Leyes de escalamiento (*scaling laws*) |
|---|---|
| **Qué dicen** | La pérdida baja como una **ley de potencia** al crecer parámetros (N), datos (D) y cómputo (C) |
| **Cómo se ven** | Rectas en gráficas log-log |
| **Fórmulas útiles** | `pérdida ≈ irreducible + a·N^(−α)` · `C ≈ 6·N·D` · Chinchilla: `D ≈ 20·N` |
| **Para qué sirven** | Predecir el resultado de un modelo grande con modelos chicos · Repartir un presupuesto entre N y D |
| **✅ Fortalezas** | Muy regulares en muchos órdenes de magnitud · Convierten la investigación en ingeniería planificable |
| **❌ Límites** | Pérdida irreducible · Los datos de calidad se agotan · Habilidades concretas no siempre siguen la pérdida · No dicen nada del costo de uso, la seguridad o la ética |
| **Por qué el Transformer** | Entrenamiento paralelo (sin cadena de cubetas) + bloques simples que se apilan y ensanchan |

## 10. 🧠 Resumen en 5 puntos
1. Los Transformers **escalan** porque se entrenan en paralelo: todos los tokens a la vez, ideal para GPUs.
2. La pérdida baja como una **ley de potencia**: una **recta en log-log**; cada mejora cuesta **multiplicar** el tamaño.
3. En tu experimento, la pendiente fue ≈ −0.06 y la recta predijo un modelo 4 veces más grande con ≈3 % de error.
4. Escalar **una sola cosa** se acaba: el modelo chico se estanca aunque le des más datos; hay que crecer N, D y C juntos.
5. **Chinchilla:** ≈20 tokens por parámetro para aprovechar el cómputo de entrenamiento; hoy se entrena con más para abaratar el uso.

---
## 11. 🏋️ Ejercicios finales

### 11.1 Repaso rápido (opción múltiple)

**1.** En una gráfica log-log, una ley de potencia se ve como…
- a) una parábola · b) una recta · c) una curva en forma de S · d) puntos al azar

**2.** Si α = 0.076, para bajar la pérdida un 10 % necesitas…
- a) 10 % más parámetros · b) el doble de parámetros · c) unas 4 veces más parámetros · d) 100 veces más parámetros

**3.** Según Chinchilla, un modelo de 1,000 millones de parámetros debería entrenarse con…
- a) 1,000 millones de tokens · b) 20,000 millones de tokens · c) 6,000 millones de tokens · d) 1 billón de tokens

**4.** Tu modelo chico ya no mejora aunque le des 10 veces más datos. Lo más probable es que…
- a) le falte capacidad (parámetros) · b) los datos estén mal · c) la ley de escalamiento no aplique · d) la tasa de aprendizaje sea perfecta

**5.** ¿Cuál **no** es una razón por la que los Transformers escalan mejor que las RNN?
- a) Procesan todos los tokens en paralelo · b) Sus bloques se apilan y ensanchan fácilmente · c) Su atención cuesta n² · d) Entrenan todas las predicciones de una frase a la vez

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La pendiente es el exponente.
2. **c)** (0.9)^(−1/0.076) ≈ 4 (Ejercicio 3).
3. **b)** 20 × 1,000 millones.
4. **a)** Es el estancamiento de la curva de d = 16 en el Ejercicio 6.
5. **c)** El costo n² es una **desventaja**; escalan bien *a pesar* de ella.
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** Se estima que el texto de buena calidad disponible en internet es del orden de decenas de billones de tokens. Con la regla de Chinchilla, ¿qué tamaño de modelo "agotaría" 20 billones de tokens? ¿Qué opciones tendrán los laboratorios cuando se acaben los datos?

**B. Caso 💼** Tu jefa quiere entrenar un modelo propio "tan bueno como los grandes" con un presupuesto 1,000 veces menor. Usa lo que aprendiste (Ejercicio 3 y Sección 6) para explicarle qué puede esperar y proponle una alternativa.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** D = 20·N → N = 20 billones / 20 = **1 billón de parámetros** (10¹²). Opciones cuando se acaben los datos: repetir datos varias **épocas** (funciona, con rendimientos decrecientes), **datos sintéticos** generados por otros modelos, otras modalidades (imágenes, video, audio), datos de mejor **calidad** en lugar de más cantidad, y gastar más cómputo al **usar** el modelo (razonar más antes de responder) en lugar de solo al entrenarlo.

**B.** Con 1,000 veces menos cómputo, la ley dice que la pérdida será claramente peor: la mejora es predecible, pero cada paso cuesta **multiplicar**. No llegará "al nivel de los grandes". Alternativa razonable: tomar un modelo **ya preentrenado** (abierto) y **ajustarlo** con sus datos (*fine-tuning*, Lección 13.7) o usar RAG (Lección 13.8). Así aprovecha el cómputo que otros ya gastaron.
</details>

### 11.3 🧩 Práctica integradora: planear el presupuesto de una startup

Una startup tiene acceso a **8 GPU durante 30 días** y quiere entrenar el mejor modelo de lenguaje posible siguiendo Chinchilla. Lo harás en 5 pasos.

**Paso 1 ▶️ — ¿Cuánto cómputo hay?** Cada GPU moderna hace del orden de 10¹⁵ operaciones por segundo en teoría, pero en la práctica se aprovecha ≈40 %.

In [ ]:
gpus, dias = 8, 30                        # ✏️
ops_por_segundo = 1e15 * 0.40             # por GPU, aprovechando el 40 %
C_total = gpus * dias * 24 * 3600 * ops_por_segundo
print(f"Presupuesto de cómputo: C ≈ {C_total:.1e} operaciones")

<details><summary><b>👀 Ver respuesta</b></summary>

C ≈ **8.3 × 10²¹** operaciones. (8 GPU × 2.6 millones de segundos × 4 × 10¹⁴ operaciones por segundo.)
</details>

**Paso 2 🔮 — Reparte entre N y D.** Combina las dos reglas: C = 6·N·D y D = 20·N. Despeja N en papel (pista: C = 6·N·20·N = 120·N²). ¿De qué tamaño será el modelo?

✍️ *Mi cálculo:*

In [ ]:
N_opt = np.sqrt(C_total / 120)
D_opt = 20 * N_opt
print(f"Modelo: N ≈ {N_opt:.1e} parámetros ({N_opt / 1e9:.1f} mil millones)")
print(f"Datos : D ≈ {D_opt:.1e} tokens ({D_opt / 1e9:,.0f} mil millones)")

<details><summary><b>👀 Ver respuesta</b></summary>

N = √(C / 120) ≈ **8.3 × 10⁹** (≈8.3 mil millones de parámetros) y D ≈ **1.7 × 10¹¹** tokens (≈166 mil millones). Con ese presupuesto, Chinchilla recomienda un modelo de unos **8 mil millones** de parámetros: el tamaño "pequeño" típico de los modelos abiertos.
</details>

**Paso 3 🔮 — ¿Y si duplican las GPU?** Antes de ejecutar: si el cómputo se **duplica**, ¿el modelo óptimo se duplica también?

In [ ]:
for factor in [1, 2, 4, 100]:            # ✏️
    N_f = np.sqrt(factor * C_total / 120)
    print(f"Cómputo ×{factor:<3} → modelo ×{N_f / N_opt:.2f} ({N_f / 1e9:.1f} mil millones) y datos ×{N_f / N_opt:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

No: con el doble de cómputo, el modelo crece solo **×1.41** (√2) y los datos también ×1.41. Con ×4 de cómputo, el modelo crece ×2; con ×100, ×10. Chinchilla dice: **reparte el cómputo extra a partes iguales** (en multiplicación) entre tamaño y datos.
</details>

**Paso 4 🔮 — ¿Qué pérdida esperar?** Supón que, con modelos pequeños, la startup midió la ley `pérdida ≈ 1.7 + 400 · N^(−0.34)` (con un término irreducible de 1.7, inventado para el ejercicio). Calcula la pérdida para su modelo y para uno 10 veces más grande.

In [ ]:
def perdida_ley(N):
    return 1.7 + 400 * N ** -0.34

for N_ in [N_opt, 10 * N_opt, 100 * N_opt]:
    print(f"N = {N_:.1e} → pérdida esperada {perdida_ley(N_):.3f}")

✋ ¿Cuánto mejora la pérdida al hacer el modelo 10 veces más grande? ¿Y 100 veces?

<details><summary><b>👀 Ver respuesta</b></summary>

- Modelo de la startup: ≈**1.87**.
- ×10: ≈**1.78** (−0.09).
- ×100: ≈**1.74** (solo −0.04 más).

Cada multiplicación por 10 da una mejora **más pequeña** porque la pérdida se acerca al piso irreducible (1.7). Las leyes de escalamiento no prometen mejoras infinitas.
</details>

**Paso 5 ✋ — Tu recomendación.** En 3 líneas: ¿qué tamaño de modelo y cuántos datos recomiendas? ¿Considerarías un modelo más chico entrenado con más datos? ¿Por qué?

<details><summary><b>👀 Ver una respuesta modelo</b></summary>

1. Según Chinchilla, un modelo de ≈8 mil millones de parámetros con ≈166 mil millones de tokens aprovecha mejor las 8 GPU durante 30 días.
2. Si el modelo se va a **usar muchísimo** (millones de consultas), conviene uno más **chico** (por ejemplo, 3 mil millones) con más tokens: cuesta un poco más de pérdida, pero es más barato y rápido de servir.
3. Antes de lanzar el entrenamiento completo, validaría la ley con 4–5 modelos pequeños (como hiciste en la Sección 3) para no tirar a la basura un mes de GPU.
</details>

---
## 📚 Para profundizar
- 📄 **Artículos:** Kaplan y colegas (2020), *Scaling Laws for Neural Language Models* · Hoffmann y colegas (2022), *Training Compute-Optimal Large Language Models* (Chinchilla).
- 📄 **Debate de las habilidades emergentes:** Wei y colegas (2022), *Emergent Abilities of Large Language Models* · Schaeffer y colegas (2023), *Are Emergent Abilities of Large Language Models a Mirage?*
- 📺 **Video:** Andrej Karpathy, *Intro to Large Language Models* (inglés, 1 h): escalamiento explicado para todo público.
- 📊 **Datos:** *Epoch AI* publica gráficas actualizadas del cómputo usado por los modelos más famosos.

---
⬅️ **Anterior:** [12.4 Encoder (BERT) vs. Decoder (GPT) vs. Encoder-Decoder (T5)](12.4-encoder-decoder.ipynb) · ➡️ **Siguiente:** [12.6 Proyecto del módulo: El aprendiz de Cervantes ✒️](12.6-proyecto-del-modulo.md)